# 02 --- Context Isolation

**CCA Pattern**: Subagents do NOT inherit coordinator context. Each starts blank
-- it receives ONLY what was explicitly passed.

This is the concept that trips up more candidates than any other. This
notebook runs three subagents through the real agent loop with a recording
client and inspects **what each one was actually sent**.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
sys.path.insert(0, str(Path('.').resolve()))

In [ ]:
from research_agents.models.research import SubTask
from research_agents.agent.context_builder import build_subagent_context
from research_agents.agent.agent_loop import AgentResult, run_agent_loop
from research_agents.agent.subagents import SUBAGENT_CONFIGS
from research_agents.services.container import make_default_services
from research_agents.testing import scripted_client, text_turn

services = make_default_services()
web_config = SUBAGENT_CONFIGS['web_researcher']

## How Context Isolation Works

The `build_subagent_context()` function in `agent/context_builder.py` is the
only way `run_coordinator()` creates subagent input. It builds a plain string
from three sources:

1. **`task.instruction`** -- what to do (always present)
2. **`task.context`** -- explicitly selected facts from the coordinator
3. **Predecessor results** -- filtered to only `task.depends_on` task IDs

What the subagent does **NOT** receive:
- The coordinator's message history
- The coordinator's system prompt
- Other subagents' results (unless listed in `depends_on`)
- The original research query (unless the coordinator chose to include it)

This is **structural isolation** -- the function signature makes it impossible
to accidentally leak coordinator state:

```python
def build_subagent_context(
    task: SubTask,
    predecessor_results: dict[str, AgentResult] | None = None,
) -> str:
```

The coordinator's `messages` list is not even a parameter. You cannot pass it.
(`run_agent_loop()` itself accepts any string as the user message -- that is
the door the anti-pattern below walks through.)

## Worked Example: Why Subagents Return MLA When Told to Use APA

This is the canonical CCA exam scenario for context isolation.

1. A user asks the coordinator: *"Research renewable energy adoption. Use
APA citation format for all sources."*
2. The coordinator reasons: "I'll decompose this into web research,
document analysis, and fact checking."
3. The coordinator creates `SubTask` objects. Each `instruction` says
"Search for renewable energy data" (or similar). **The APA requirement
never makes it into any `SubTask.context`.**
4. The web researcher runs, does a fine job finding sources, and returns
citations formatted however it defaults -- MLA, say.
5. The user sees the final report, notices the citations are wrong, and
reports it as a bug.

**Why the exam calls this the most-missed pattern:** it looks like the
subagent disobeyed the coordinator. It didn't. It never received the
instruction. The APA requirement was in the coordinator's turn-1 message,
which the subagent structurally cannot see.

**The fix is explicit forwarding.** Anything that applies to a subagent
(citation format, tone, date range, output schema) must be copied into its
`SubTask.context`. No inheritance. No magic.

The tempting *wrong* fix is to hand the subagent the whole coordinator
conversation "so it can't miss anything". That is the anti-pattern, and the
cells below measure what it actually sends.

### The coordinator history used in every cell below

Three messages: the user's request (with the APA rule), the coordinator's
own reasoning, and a tool result that came back from a *different*
subagent. What a subagent should and should not see is now concrete.

In [ ]:
coordinator_messages = [
    {'role': 'user', 'content': 'Research renewable energy adoption globally. '
                                'Use APA citation format for all sources.'},
    {'role': 'assistant', 'content': 'I will decompose this into web research, '
                                     'document analysis, and fact checking.'},
    {'role': 'user', 'content': '[tool_result from document_analyzer t2] '
                                'IEA report: solar capacity reached 1580 GW in 2024.'},
]

APA = 'APA citation format'          # the instruction the subagent must see
REASONING = 'I will decompose'       # coordinator's private reasoning
OTHER_AGENT = '1580 GW'              # another subagent's result

def sent_to_subagent(client) -> str:
    """The user message the subagent actually received on its first turn."""
    return client.calls[0]['messages'][0]['content']

## Anti-Pattern: Shared Context

`run_leaky_subagent()` in `anti_patterns/shared_context.py` passes the
coordinator's full message history to the subagent. We run it through the
real loop with a recording client and read back what was sent.

In [ ]:
from research_agents.anti_patterns.shared_context import run_leaky_subagent

leaky_client = scripted_client([text_turn('Found 3 sources.')])
run_leaky_subagent(leaky_client, services, coordinator_messages, 'web_researcher')

leaked_context = sent_to_subagent(leaky_client)
print(f'Leaked context length: {len(leaked_context)} chars')
print(f'Has citation instruction:    {APA in leaked_context}')
print(f'Has coordinator reasoning:   {REASONING in leaked_context}')
print(f"Has another agent's result:  {OTHER_AGENT in leaked_context}")
print()
print(leaked_context[:160] + '...')

### Why This Fails

The subagent did get the APA rule -- along with everything else. The cell
shows three problems in the one string it received:

- **Context pollution**: the web researcher sees the coordinator's private
planning ("I will decompose...") and a document analyzer's tool result. Neither
is its job, and both compete for attention with its actual task.
- **No isolation**: whatever any other subagent produced is now visible to
this one, so a mistake upstream propagates everywhere.
- **Wrong format**: `str(coordinator_messages)` is a Python `repr` of a list of
dicts, not a task. The subagent has to parse role markers to find its
instruction.

Token waste scales with conversation length: a three-message history is short
here, but a real coordinator history is thousands of tokens the subagent
re-reads on every turn.

## Correct Pattern: Explicit Context Passing

The context builder passes ONLY what the coordinator deliberately selects.
That cuts both ways, so we run it twice.

### Case 1 -- the exam's bug: the coordinator forgot to forward APA

In [ ]:
forgot_task = SubTask(
    task_id='t1',
    agent_type='web_researcher',
    instruction='Search for renewable energy adoption statistics',
    context='Focus on 2024 data from government and peer-reviewed sources.',
)
forgot_client = scripted_client([text_turn('Found 3 sources.')])
run_agent_loop(forgot_client, services, build_subagent_context(forgot_task),
               web_config.system_prompt, web_config.tools, 'web_researcher')

forgot_context = sent_to_subagent(forgot_client)
print(forgot_context)
print()
print(f'Has citation instruction: {APA in forgot_context}')

The subagent received a clean task and **no citation rule**. Whatever format it
returns is not disobedience; the rule was never in its input. This is the
exact failure the exam describes -- and note that nothing in the code raised
an error, because from the code's point of view the coordinator did exactly
what it asked.

### Case 2 -- the fix: forward the rule in `SubTask.context`

In [ ]:
task = SubTask(
    task_id='t1',
    agent_type='web_researcher',
    instruction='Search for renewable energy adoption statistics',
    context='Focus on 2024 data from government and peer-reviewed sources. '
            'Use APA citation format.',
)
explicit_client = scripted_client([text_turn('Found 3 sources.')])
run_agent_loop(explicit_client, services, build_subagent_context(task),
               web_config.system_prompt, web_config.tools, 'web_researcher')

explicit_context = sent_to_subagent(explicit_client)
print(f'Explicit context length: {len(explicit_context)} chars')
print(f'Has citation instruction:    {APA in explicit_context}')
print(f'Has coordinator reasoning:   {REASONING in explicit_context}')
print(f"Has another agent's result:  {OTHER_AGENT in explicit_context}")
print()
print(explicit_context)

### Predecessor Results: The `depends_on` Filter

When a task depends on earlier tasks, the context builder includes only those
specific results -- not all prior results:

In [ ]:
# Simulate predecessor results from Wave 0
prior_results = {
    't1': AgentResult(content='Found 4 sources on renewable energy.'),
    't2': AgentResult(content='Database shows 1580 GW solar capacity in 2024.'),
    't3': AgentResult(content='Document analysis of IEA report complete.'),
}

# Fact checker depends on t1 and t2 only -- NOT t3
fact_check_task = SubTask(
    task_id='t4',
    agent_type='fact_checker',
    instruction='Verify renewable energy claims',
    context='Cross-reference web and database findings',
    depends_on=['t1', 't2'],  # Only these results are passed
)

context_with_deps = build_subagent_context(fact_check_task, prior_results)
print(context_with_deps)
print()
print(f'Contains t1 result: {"Found 4 sources" in context_with_deps}')
print(f'Contains t2 result: {"1580 GW" in context_with_deps}')
print(f'Contains t3 result: {"IEA report" in context_with_deps}')

### Comparison

Every value below is read from what the recording client captured, i.e. the
user message each subagent was actually sent. Boolean rows are phrased as
properties the correct pattern should have, so `FIXED` means the fix is
present.

In [ ]:
from helpers import compare_results

print('Anti-pattern (leaky) vs. correct (explicit, APA forwarded):')
compare_results(
    {'context_length': len(leaked_context),
     'has_citation_instruction': APA in leaked_context,
     'free_of_coordinator_reasoning': REASONING not in leaked_context,
     'free_of_other_agent_results': OTHER_AGENT not in leaked_context,
     'sent_as': 'repr(messages)'},
    {'context_length': len(explicit_context),
     'has_citation_instruction': APA in explicit_context,
     'free_of_coordinator_reasoning': REASONING not in explicit_context,
     'free_of_other_agent_results': OTHER_AGENT not in explicit_context,
     'sent_as': 'task string'},
)
print()
print("The exam's bug (rule not forwarded) vs. the fix (rule forwarded):")
compare_results(
    {'has_citation_instruction': APA in forgot_context},
    {'has_citation_instruction': APA in explicit_context},
)

## CCA Exam Tip

> Any question where a subagent produces results that 'should have followed the
coordinator's instructions' is testing context isolation.
> - The answer is always that the instructions were in the coordinator's context
but never explicitly forwarded
> - Subagents do not inherit. Subagents receive only what you explicitly send.
> - The `build_subagent_context()` pattern enforces this structurally --
the coordinator's messages are never a function parameter
> - "Send the whole history" is the distractor: it forwards the rule *and*
everything else, as the leaky run above shows

What this notebook does **not** test: whether the model *obeys* an APA rule
once it sees one. That is model behaviour. What it measures is what the model
can see, which is the part architecture controls.